# Phase H: Analysis & Thesis Figures

Generate all figures and tables for thesis Chapter 5 and defense.

**Prerequisites:** Phases C, D, E, F all complete.


## 1. Environment Setup


In [ ]:
# Install dependencies
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    print('WARNING: No GPU detected')


## 1b. Copy GLADA Package to Local Runtime


In [ ]:
import shutil
import os
import sys

# Define paths
DRIVE_PKG = "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada"
LOCAL_PKG = '/content/glada'

# Remove existing local package if it exists to ensure a fresh copy
if os.path.exists(LOCAL_PKG):
    shutil.rmtree(LOCAL_PKG)

# Copy the directory
try:
    shutil.copytree(DRIVE_PKG, LOCAL_PKG)
    print(f'Successfully copied package to {LOCAL_PKG}')
except Exception as e:
    print(f'Error copying package: {e}')

# Add to sys.path
if LOCAL_PKG not in sys.path:
    sys.path.insert(0, LOCAL_PKG)

# Verify
print(f'Package copied to {LOCAL_PKG}')
print(f"Contents of {LOCAL_PKG}: {os.listdir(LOCAL_PKG)}")


In [ ]:
# ============================================================================
# 1.0 Environment Setup — Drive mount, imports, matplotlib config
# ============================================================================

import os
import json
import glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats

# --- Matplotlib defaults for thesis-quality figures ---
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,              # Sharp in-notebook display
    'savefig.dpi': 300,             # Print-quality exports
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'axes.spines.top': False,       # Remove top spine
    'axes.spines.right': False,     # Remove right spine
    'axes.grid': False,             # No gridlines by default
    'figure.constrained_layout.use': True,
})
# bbox_inches='tight' is set per-savefig call, not as rcParam

# Accessible colorblind palette (Okabe & Ito, 2008)
PALETTE = sns.color_palette('colorblind')
sns.set_palette('colorblind')

# Named colors for consistent use across all figures
COLOR_PRIMARY = PALETTE[0]     # Blue — main model / full GLADA
COLOR_SECONDARY = PALETTE[1]   # Orange — baselines / ablation removals
COLOR_ACCENT = PALETTE[2]      # Green — improvements / positive deltas
COLOR_WARN = PALETTE[3]        # Red — degradation / negative deltas
COLOR_NEUTRAL = PALETTE[4]     # Purple — neutral / reference lines

# --- Paths ---
EXPERIMENTS_DIR = '/content/drive/MyDrive/Jakob/Experiments'
SAVE_DIR = os.path.join(EXPERIMENTS_DIR, 'figures')
os.makedirs(SAVE_DIR, exist_ok=True)

ABLATION_FILE = os.path.join(EXPERIMENTS_DIR, 'ablation', 'ablation_results.json')
COMPREHENSIVE_FILE = os.path.join(EXPERIMENTS_DIR, 'comprehensive_eval', 'results.json')

print(f"Experiments directory: {EXPERIMENTS_DIR}")
print(f"Figures will be saved to: {SAVE_DIR}")
print(f"Ablation results: {ABLATION_FILE} — exists: {os.path.exists(ABLATION_FILE)}")
print(f"Comprehensive eval: {COMPREHENSIVE_FILE} — exists: {os.path.exists(COMPREHENSIVE_FILE)}")
print(f"Palette colors: {len(PALETTE)} colorblind-safe colors loaded")

## 2. Load Experiment Data

Auto-discover all GLADA experiment runs from Drive. Each run directory follows the convention:

```
/content/drive/MyDrive/Jakob/Experiments/
  glada_{backbone}_{variant}/
    {run_id}/
      history.json          <- list of per-epoch dicts with all training metrics
      config.json           <- full GLADAConfig used for that run
      multi_eval_results.json  <- cross-dataset eval (if glada_evaluation.ipynb was run)
      best_model.pt         <- model.state_dict() of best checkpoint
```

The `load_run()` helper loads all available JSON files from a run directory. Missing files are silently skipped (e.g., `multi_eval_results.json` only exists after running the evaluation notebook).

In [ ]:
# ============================================================================
# 2.0 Load All Experiment Data — auto-discovery from Drive
# ============================================================================

# --- Load comprehensive evaluation results (16 models x 6 datasets) ---
# This is the primary data source for cross-dataset analysis, backbone
# comparison, ablation charts, and SOTA tables.
comp_results = {}
if os.path.exists(COMPREHENSIVE_FILE):
    with open(COMPREHENSIVE_FILE) as f:
        comp_results = json.load(f)
    print(f"Loaded comprehensive eval: {len(comp_results)} models")
    for name, entry in comp_results.items():
        datasets = list(entry.get('datasets', {}).keys())
        print(f"  {name}: backbone={entry.get('backbone','?')}, datasets={datasets}")
else:
    print(f"WARNING: Comprehensive eval not found at {COMPREHENSIVE_FILE}")

# --- Helper: flatten nested history.json into per-epoch dicts ---
# history.json uses nested structure:
#   h['train']['spoof_loss'], h['eval']['ITW']['eer'], etc.
# We flatten to: h['spoof_loss'], h['itw_eer'], etc. for plotting convenience.
def flatten_history(history_list):
    """Flatten nested history.json entries into flat per-epoch dicts."""
    flat = []
    for h in history_list:
        row = {
            'epoch': h.get('epoch'),
            'phase': h.get('phase'),
            'lr': h.get('lr'),
            'domain_alpha': h.get('domain_alpha'),
            'speaker_alpha': h.get('speaker_alpha'),
            'kl_loss_raw': h.get('kl_loss_raw'),
            'kl_loss_clamped': h.get('kl_loss_clamped'),
        }
        # Flatten train metrics: h['train']['spoof_loss'] -> row['spoof_loss']
        train = h.get('train', {})
        for k, v in train.items():
            row[k] = v
        # Flatten eval metrics: h['eval']['ITW']['eer'] -> row['itw_eer']
        for ds_name, ds_metrics in h.get('eval', {}).items():
            ds_key = ds_name.lower().replace(' ', '_')
            for metric, val in ds_metrics.items():
                if metric == 'name':
                    continue
                row[f'{ds_key}_{metric}'] = val
        flat.append(row)
    return flat


def load_run(run_dir):
    """Load history.json and config.json from a run directory.

    Returns dict with keys 'history' (flattened), 'config', 'dir'.
    """
    result = {'dir': run_dir}

    history_path = os.path.join(run_dir, 'history.json')
    if os.path.exists(history_path):
        with open(history_path) as f:
            raw = json.load(f)
        result['history'] = flatten_history(raw)

    config_path = os.path.join(run_dir, 'config.json')
    if os.path.exists(config_path):
        with open(config_path) as f:
            result['config'] = json.load(f)

    return result


# --- Auto-discover all runs under ablation_* and multiseed_* directories ---
# When multiple timestamped subdirectories exist (e.g., 20260305_214705,
# 20260306_022041), only load the MOST RECENT one — older runs are stale
# re-runs that may have incomplete training history.
all_runs = {}
run_patterns = ['ablation_*', 'multiseed_*']
for pattern in run_patterns:
    for exp_dir in sorted(glob.glob(os.path.join(EXPERIMENTS_DIR, pattern))):
        exp_name = os.path.basename(exp_dir)
        # Check if history.json is directly in the exp dir (flat layout)
        if os.path.exists(os.path.join(exp_dir, 'history.json')):
            data = load_run(exp_dir)
            all_runs[exp_name] = data
        else:
            # Timestamped subdirectories — take only the most recent one
            subdirs = sorted(glob.glob(os.path.join(exp_dir, '*')))
            subdirs = [d for d in subdirs if os.path.isdir(d)
                       and os.path.exists(os.path.join(d, 'history.json'))]
            if subdirs:
                latest = subdirs[-1]  # sorted lexicographically = chronological for YYYYMMDD_HHMMSS
                data = load_run(latest)
                all_runs[exp_name] = data
                if len(subdirs) > 1:
                    skipped = [os.path.basename(d) for d in subdirs[:-1]]
                    print(f"  {exp_name}: {len(subdirs)} runs found, using latest "
                          f"({os.path.basename(latest)}), skipped: {skipped}")

# --- Print discovery summary ---
print(f"\nDiscovered {len(all_runs)} experiment runs with training history:")
print("=" * 80)
for name, run in all_runs.items():
    h = run.get('history', [])
    cfg = run.get('config', {})
    backbone = cfg.get('backbone', 'unknown')
    seed = cfg.get('seed', '?')
    use_glada = cfg.get('use_glada', '?')
    # Get best ITW EER from flattened history
    itw_eers = [ep.get('itw_eer') for ep in h if ep.get('itw_eer') is not None]
    best_itw = f"{min(itw_eers):.2f}%" if itw_eers else "N/A"
    print(f"  {name}: {len(h)} epochs | {backbone} | seed={seed} | glada={use_glada} | best_itw={best_itw}")

if not all_runs:
    print("  (No runs found — check that ablation_*/multiseed_* dirs are synced to Drive)")
    print(f"  Looking in: {EXPERIMENTS_DIR}")
    print(f"  Dirs found: {glob.glob(os.path.join(EXPERIMENTS_DIR, '*'))[:10]}")

## 3. Training Curves

Visualize training dynamics across the two-phase training protocol:

- **Phase 1** (epochs 1--5): Frozen backbone, head-only training at high LR. This lets the classification head, domain discriminators, and speaker adversary warm up before the backbone is unfrozen.
- **Phase 2** (epochs 6--50): Unfrozen backbone with LLRD (Clark et al., EMNLP 2020), lower backbone LR. Gradient checkpointing enabled to fit in T4 memory.

**Key signals to monitor:**

| Metric | Expected Behavior | Interpretation |
|---|---|---|
| Spoof loss | Steady decrease | Task learning is progressing |
| Domain/GLADA loss | Initial increase, then plateau | Backbone learning domain-invariant features |
| Speaker accuracy | Decrease over training | Speaker adversary stripping identity (via GRL) |
| GRL alpha | 0 -> max over warmup | Ganin et al. (JMLR, 2016) gradual scheduling |
| KL divergence | Small, stable | VIB bottleneck not collapsing (Alemi et al., ICLR 2017) |
| CKA loss | Decrease | Layer groups becoming more diverse (Kornblith et al., ICML 2019) |

The vertical dashed line marks the Phase 1 -> Phase 2 boundary (backbone unfreezing).

In [ ]:
# ============================================================================
# 3.0 Training Curves — 2x3 subplot grid of training dynamics
# ============================================================================

def plot_training_curves(history, title="Training Curves", save_name=None,
                         phase_boundary=5):
    """Plot 2x3 grid of training dynamics from flattened history.

    Subplots:
      (0,0) Main losses: spoof_loss, total_loss
      (0,1) Component losses: glada_loss, domain_loss, speaker_loss, cka_loss, kl_loss
      (0,2) EER per eval dataset
      (1,0) Accuracies: spoof_acc, speaker_acc
      (1,1) Learning rate schedule
      (1,2) GRL alpha warmup schedule
    """
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    epochs = [h['epoch'] for h in history]

    def add_phase_line(ax):
        ax.axvline(x=phase_boundary, color='gray', linestyle='--', alpha=0.5,
                   linewidth=1, label='Phase 1/2 boundary')

    # --- (0,0) Main losses ---
    ax = axes[0, 0]
    for i, loss_name in enumerate(['spoof_loss', 'total_loss']):
        vals = [h.get(loss_name) for h in history]
        if any(v is not None for v in vals):
            ax.plot(epochs, vals, label=loss_name.replace('_', ' ').title(),
                    color=PALETTE[i], linewidth=2)
    add_phase_line(ax)
    ax.set_title('Main Losses')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)

    # --- (0,1) Component losses ---
    ax = axes[0, 1]
    component_losses = ['glada_loss', 'domain_loss', 'speaker_loss', 'cka_loss', 'kl_loss']
    for i, loss_name in enumerate(component_losses):
        vals = [h.get(loss_name) for h in history]
        if any(v is not None for v in vals):
            vals_clean = [v if v is not None else float('nan') for v in vals]
            ax.plot(epochs, vals_clean, label=loss_name.replace('_', ' ').title(),
                    color=PALETTE[i], linewidth=1.5)
    add_phase_line(ax)
    ax.set_title('Component Losses')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)

    # --- (0,2) EER per eval dataset ---
    ax = axes[0, 2]
    # Flattened keys are like 'itw_eer', '2019_la_eer', etc.
    eval_keys = sorted([k for k in history[0] if k.endswith('_eer')])
    for i, key in enumerate(eval_keys):
        vals = [h.get(key) for h in history]
        if any(v is not None for v in vals):
            # EER values from comprehensive_eval are already in %, but
            # history.json eval EERs are also in % (checked: 20.28 not 0.2028)
            vals_clean = [v if v is not None else float('nan') for v in vals]
            label = key.replace('_eer', '').upper()
            ax.plot(epochs, vals_clean, label=label, color=PALETTE[i % len(PALETTE)], linewidth=2)
    add_phase_line(ax)
    ax.set_title('EER per Dataset')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('EER (%)')
    ax.legend(fontsize=9)

    # --- (1,0) Accuracies ---
    ax = axes[1, 0]
    for i, acc_name in enumerate(['spoof_acc', 'speaker_acc']):
        vals = [h.get(acc_name) for h in history]
        if any(v is not None for v in vals):
            vals_clean = [v if v is not None else float('nan') for v in vals]
            ax.plot(epochs, vals_clean, label=acc_name.replace('_', ' ').title(),
                    color=PALETTE[i], linewidth=2)
    add_phase_line(ax)
    ax.set_title('Accuracies')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9)

    # --- (1,1) Learning rate schedule ---
    ax = axes[1, 1]
    lr_vals = [h.get('lr') for h in history]
    if any(v is not None for v in lr_vals):
        lr_clean = [v if v is not None else float('nan') for v in lr_vals]
        ax.plot(epochs, lr_clean, color=COLOR_PRIMARY, linewidth=2, label='Head LR')
    add_phase_line(ax)
    ax.set_title('Learning Rate Schedule')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.ticklabel_format(style='scientific', axis='y', scilimits=(0, 0))
    ax.legend(fontsize=9)

    # --- (1,2) GRL alpha warmup schedule ---
    ax = axes[1, 2]
    for i, alpha_name in enumerate(['domain_alpha', 'speaker_alpha']):
        vals = [h.get(alpha_name) for h in history]
        if any(v is not None for v in vals):
            vals_clean = [v if v is not None else float('nan') for v in vals]
            ax.plot(epochs, vals_clean,
                    label=alpha_name.replace('_', ' ').title(),
                    color=PALETTE[i], linewidth=2)
    add_phase_line(ax)
    ax.set_title('GRL Alpha Schedule')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Alpha')
    ax.legend(fontsize=9)

    fig.suptitle(title, fontsize=18, y=1.02)

    if save_name:
        save_path = os.path.join(SAVE_DIR, save_name)
        fig.savefig(save_path)
        print(f"Saved: {save_path}")
    plt.show()


# --- Plot training curves for each loaded run ---
if all_runs:
    for run_name, run_data in all_runs.items():
        history = run_data.get('history', [])
        if not history:
            continue
        cfg = run_data.get('config', {})
        phase_b = cfg.get('freeze_epochs', cfg.get('phase1_epochs', 5))
        plot_training_curves(
            history,
            title=f"Training Curves: {run_name}",
            save_name=f"training_curves_{run_name.replace('/', '_')}.png",
            phase_boundary=phase_b,
        )
else:
    print("No runs loaded — training curves skipped.")

## 4. Backbone Comparison

Compare best In-the-Wild (ITW) EER across SSL backbones (Muller et al., Interspeech 2022). When multiple seeds are available, we show mean +/- standard deviation with individual data points overlaid as a strip plot.

**Backbones tested:**

| Backbone | Params | Pretraining | Source |
|---|---|---|---|
| WavLM-Large | 317M | 94K hrs speech | Microsoft (Chen et al., 2022) |
| XLSR-300M | 300M | 128 languages | Meta (Babu et al., 2022) |
| Data2Vec-Audio-Large | 314M | Self-supervised prediction | Meta (Baevski et al., 2022) |
| Wav2Vec2-Large-Robust | 317M | Diverse acoustic conditions | Meta (Hsu et al., 2021) |
| WavLM-Base-Plus | 95M | 94K hrs speech (smaller) | Microsoft (Chen et al., 2022) |

WavLM-Large is expected to perform best based on prior work in audio deepfake detection (El-Kheir et al., NAACL 2025; Liang et al., 2026).

In [ ]:
# ============================================================================
# 4.0 Backbone Comparison — bar chart from comprehensive eval results
# ============================================================================

def plot_backbone_comparison(comp_data, save_name=None):
    """Compare ITW EER across backbone variants using comprehensive_eval/results.json.

    Selects entries whose config name starts with 'bb_' (backbone runs) plus
    'full_glada' as the wavlm-base-plus reference.
    """
    # --- Extract backbone entries ---
    backbone_keys = [k for k in comp_data if k.startswith('bb_')]
    backbone_keys.append('full_glada')  # Reference row

    rows = []
    for key in backbone_keys:
        if key not in comp_data:
            continue
        entry = comp_data[key]
        itw = entry.get('datasets', {}).get('ITW', {})
        rows.append({
            'Config': key,
            'Backbone': entry.get('backbone', 'unknown'),
            'ITW EER (%)': itw.get('eer', float('nan')),
            'CI Low': itw.get('ci_low', float('nan')),
            'CI High': itw.get('ci_high', float('nan')),
        })

    if not rows:
        print("No backbone data in comprehensive results.")
        return None

    df = pd.DataFrame(rows).sort_values('ITW EER (%)')

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(df))

    # Error bars from bootstrap CI
    ci_lower = df['ITW EER (%)'] - df['CI Low']
    ci_upper = df['CI High'] - df['ITW EER (%)']
    yerr = np.array([ci_lower.values, ci_upper.values])
    # Clamp negative errors to 0 (can happen with NaN CIs)
    yerr = np.nan_to_num(yerr, nan=0.0)
    yerr = np.clip(yerr, 0, None)

    colors = [COLOR_ACCENT if c == 'full_glada' else PALETTE[i % len(PALETTE)]
              for i, c in enumerate(df['Config'])]

    bars = ax.bar(x, df['ITW EER (%)'], yerr=yerr, capsize=5, width=0.6,
                  color=colors, edgecolor='none', alpha=0.85,
                  error_kw={'linewidth': 1.5, 'capthick': 1.5})

    # Value labels
    for i, (bar, eer, ci_lo, ci_hi) in enumerate(zip(
        bars, df['ITW EER (%)'], df['CI Low'], df['CI High']
    )):
        label = f"{eer:.2f}%"
        if not np.isnan(ci_lo):
            label += f"\n({ci_lo:.1f}–{ci_hi:.1f})"
        offset = yerr[1][i] if yerr[1][i] > 0 else 0.3
        ax.text(bar.get_x() + bar.get_width() / 2.0,
                bar.get_height() + offset + 0.3,
                label, ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(df['Config'], rotation=25, ha='right')
    ax.set_ylabel('ITW EER (%)')
    ax.set_title('SSL Backbone Comparison — In-the-Wild EER')

    if save_name:
        save_path = os.path.join(SAVE_DIR, save_name)
        fig.savefig(save_path)
        print(f"Saved: {save_path}")
    plt.show()

    print("\nBackbone Summary:")
    print(df.to_markdown(index=False, floatfmt='.2f'))
    return df


if comp_results:
    backbone_df = plot_backbone_comparison(comp_results, save_name='fig_backbone_comparison.png')
else:
    print("Comprehensive results not loaded — backbone comparison skipped.")

## 5. Ablation Results

Visualize the ablation study results from `glada_ablation.ipynb`. The critical comparison is:

- **Full GLADA** (all components enabled) vs. **no_glada** (single-level DA only via global domain discriminator)

This directly tests the thesis hypothesis: **per-group domain adaptation outperforms single-level domain adaptation** (following the precedent set by MLADA (Qu et al., Neurocomputing 2022) in vision and DAA (Ryu et al., ACL 2021 Findings) in NLP).

**Color coding:**
- Green: Full GLADA (reference configuration)
- Red: Removing this component **hurts** (EER increases relative to full model)
- Blue: Removing this component **helps or is neutral** (EER decreases or unchanged)

Each ablation configuration toggles exactly one component off while keeping all others at their default values, following standard ablation methodology.

In [ ]:
# ============================================================================
# 5.0 Ablation Results — horizontal bar chart + cross-dataset heatmap
# ============================================================================

# --- Use comprehensive_eval/results.json for ablation (wavlm-base-plus configs) ---
ablation_configs = [
    'full_glada', 'no_glada', 'no_vib', 'no_speaker', 'no_cka',
    'no_rawboost', 'no_hier', 'global_da_only', 'hierarchical_only',
    'no_da', 'backbone_only',
]

ablation_data = {k: comp_results[k] for k in ablation_configs if k in comp_results}

if ablation_data:
    # --- Build DataFrame with all 6 datasets ---
    dataset_names = ['2019_LA', 'ITW', '2021_LA', '2021_DF', 'WaveFake', 'CodecFake']
    rows = []
    for name, entry in ablation_data.items():
        row = {'Config': name}
        for ds in dataset_names:
            ds_data = entry.get('datasets', {}).get(ds, {})
            row[ds] = ds_data.get('eer', float('nan'))
            if ds == 'ITW':
                row['ITW CI Low'] = ds_data.get('ci_low', float('nan'))
                row['ITW CI High'] = ds_data.get('ci_high', float('nan'))
        rows.append(row)

    df_abl = pd.DataFrame(rows).sort_values('ITW')

    # --- Print markdown table ---
    print("Ablation Study Results (EER %):")
    print(df_abl[['Config'] + dataset_names].to_markdown(index=False, floatfmt='.2f'))

    # --- Horizontal bar chart (ITW EER) ---
    full_eer = ablation_data.get('full_glada', {}).get('datasets', {}).get('ITW', {}).get('eer', 0)

    fig, ax = plt.subplots(figsize=(12, max(6, len(df_abl) * 0.6)))

    colors = []
    for _, row in df_abl.iterrows():
        if row['Config'] == 'full_glada':
            colors.append(COLOR_ACCENT)
        elif row['ITW'] > full_eer:
            colors.append(COLOR_WARN)
        else:
            colors.append(COLOR_PRIMARY)

    bars = ax.barh(df_abl['Config'], df_abl['ITW'],
                   color=colors, edgecolor='none', alpha=0.85, height=0.6)

    ax.axvline(x=full_eer, color=COLOR_ACCENT, linestyle='--', alpha=0.7, linewidth=1.5,
               label=f'Full GLADA ({full_eer:.2f}%)')

    for bar, eer_val in zip(bars, df_abl['ITW']):
        delta = eer_val - full_eer
        delta_str = f" ({delta:+.2f})" if abs(delta) > 0.01 else ""
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
                f"{eer_val:.2f}%{delta_str}", ha='left', va='center', fontsize=10)

    ax.set_xlabel('ITW EER (%)')
    ax.set_title('Ablation Study: Component Contribution to GLADA (ITW)')
    ax.legend(loc='lower right', fontsize=10)

    save_path = os.path.join(SAVE_DIR, 'fig_ablation_chart.png')
    fig.savefig(save_path)
    print(f"\nSaved: {save_path}")
    plt.show()

    # --- Cross-dataset heatmap (all 16 models x 6 datasets) ---
    print("\n--- Cross-Dataset Heatmap (all 16 models) ---")
    all_model_rows = []
    for name, entry in comp_results.items():
        row = {'Config': name}
        for ds in dataset_names:
            row[ds] = entry.get('datasets', {}).get(ds, {}).get('eer', float('nan'))
        all_model_rows.append(row)

    df_all = pd.DataFrame(all_model_rows).set_index('Config')
    # Sort by mean EER
    df_all['Mean'] = df_all.mean(axis=1)
    df_all = df_all.sort_values('Mean')
    df_all = df_all.drop(columns='Mean')

    fig, ax = plt.subplots(figsize=(12, max(8, len(df_all) * 0.5)))
    sns.heatmap(df_all, annot=True, fmt='.1f', cmap='RdYlGn_r',
                linewidths=0.5, ax=ax, cbar_kws={'label': 'EER (%)'})
    ax.set_title('Cross-Dataset EER Heatmap (16 Models x 6 Datasets)')
    ax.set_ylabel('')
    ax.set_xlabel('Evaluation Dataset')

    save_path = os.path.join(SAVE_DIR, 'fig_crossdataset_heatmap.png')
    fig.savefig(save_path)
    print(f"Saved: {save_path}")
    plt.show()

    # --- Component contribution summary ---
    print(f"\nFull GLADA ITW EER: {full_eer:.2f}%")
    print("\nComponent contributions (delta from full model, ITW):")
    print("-" * 55)
    for _, row in df_abl.iterrows():
        if row['Config'] == 'full_glada':
            continue
        delta = row['ITW'] - full_eer
        direction = '\u2191 hurts' if delta > 0 else '\u2193 helps'
        print(f"  {row['Config']:25s}: {row['ITW']:6.2f}%  ({delta:+.2f}% {direction})")
else:
    print("No ablation data in comprehensive results.")

## 6. SOTA Comparison Table

Compare GLADA against published results on standard evaluation benchmarks. EER (%) values sourced from original papers.

| Method | Paper | Key Feature | Relevance to GLADA |
|---|---|---|---|
| AASIST | Jung et al. (ICASSP, 2022) | Graph attention network on RawNet features | Strong single-model baseline |
| RawNet2 | Tak et al. (Interspeech, 2021) | Raw waveform end-to-end CNN | Classic raw waveform approach |
| IB-CAAN | Huang et al. (2025) | VIB + adversarial alignment (single-level DA) | Closest prior work; GLADA extends to per-group DA |
| MultiConv+CKA | Tran et al. (ACM MM, 2025) | Multi-kernel convolutions + CKA diversity | Uses CKA; no domain adaptation |
| HierCon | Liang et al. (2026) | Hierarchical contrastive learning on grouped layers | Uses grouped layers; no adversarial DA |
| **GLADA (Ours)** | This work | Per-group adversarial DA + VIB + hierarchical attention | Novel: per-group GRL + domain discriminators |

**Benchmark datasets:**
- **ASV19 LA**: ASVspoof 2019 Logical Access eval partition (Wang et al., 2020)
- **ASV21 LA**: ASVspoof 2021 Logical Access eval (Yamagishi et al., 2021)
- **ASV21 DF**: ASVspoof 2021 Deepfake eval (Yamagishi et al., 2021)
- **ITW**: In-the-Wild dataset (Muller et al., Interspeech 2022) -- primary cross-domain benchmark

In [ ]:
# ============================================================================
# 6.0 SOTA Comparison Table — published results + our best GLADA results
# ============================================================================

# --- Published SOTA results (EER % from original papers) ---
sota = pd.DataFrame([
    {'Method': 'RawNet2 (Tak et al., 2021)',         'ASV19 LA': 3.15,  'ASV21 LA': None, 'ASV21 DF': None, 'ITW': 37.80, 'Source': 'Interspeech 2021'},
    {'Method': 'AASIST (Jung et al., 2022)',          'ASV19 LA': 0.83,  'ASV21 LA': 9.29, 'ASV21 DF': 3.37, 'ITW': 31.80, 'Source': 'ICASSP 2022'},
    {'Method': 'XLSR + Linear ERM',                   'ASV19 LA': None,  'ASV21 LA': None, 'ASV21 DF': None, 'ITW': 14.13, 'Source': 'Huang et al. 2025'},
    {'Method': 'IB-CAAN (Huang et al., 2025)',        'ASV19 LA': 0.40,  'ASV21 LA': 2.21, 'ASV21 DF': 1.64, 'ITW': 4.93,  'Source': 'arXiv 2509.23618'},
    {'Method': 'IB-CAAN + full augmentation',          'ASV19 LA': None,  'ASV21 LA': None, 'ASV21 DF': None, 'ITW': 4.61,  'Source': 'Huang et al. 2025'},
    {'Method': '4-model fusion (Serrano et al.)',      'ASV19 LA': None,  'ASV21 LA': None, 'ASV21 DF': None, 'ITW': 6.40,  'Source': 'Serrano et al. 2025'},
    {'Method': 'HierCon (Liang et al., 2026)',        'ASV19 LA': None,  'ASV21 LA': None, 'ASV21 DF': None, 'ITW': 6.87,  'Source': 'arXiv 2602.01032'},
    {'Method': 'MultiConv+CKA (Tran et al., 2025)',   'ASV19 LA': None,  'ASV21 LA': None, 'ASV21 DF': None, 'ITW': 4.44,  'Source': 'ACM MM 2025'},
])

# --- Add our best results from comprehensive eval ---
our_best_models = [
    ('bb_wavlm-large_s42', 'GLADA (Ours) — WavLM-Large s42 [best overall]'),
    ('no_cka', 'GLADA (Ours) — no_cka [best wavlm-base-plus]'),
    ('full_glada', 'GLADA (Ours) — full_glada [proposed model]'),
]

our_rows = []
eer_cols = ['ASV19 LA', 'ASV21 LA', 'ASV21 DF', 'ITW']
dataset_map = {'ASV19 LA': '2019_LA', 'ASV21 LA': '2021_LA', 'ASV21 DF': '2021_DF', 'ITW': 'ITW'}

for config_key, display_name in our_best_models:
    if config_key not in comp_results:
        continue
    entry = comp_results[config_key]
    row = {'Method': display_name, 'Source': 'This work'}
    for col, ds_key in dataset_map.items():
        eer = entry.get('datasets', {}).get(ds_key, {}).get('eer')
        row[col] = eer if eer is not None else None
    our_rows.append(row)

if our_rows:
    full_df = pd.concat([sota, pd.DataFrame(our_rows)], ignore_index=True)
else:
    full_df = sota.copy()
    print("WARNING: No GLADA results found in comp_results.")


def format_eer(x):
    if pd.isna(x) or x is None:
        return '\u2014'
    return f'{x:.2f}'


display_df = full_df.copy()
for col in eer_cols:
    display_df[col] = display_df[col].apply(format_eer)

print("EER (%) Comparison with Published Methods:")
print("=" * 90)
print(display_df.to_markdown(index=False))

# --- LaTeX version ---
print("\n\nLaTeX table (paste into thesis):")
print("-" * 80)
print(r"\begin{table}[t]")
print(r"\centering")
print(r"\caption{EER (\%) comparison with published methods on standard benchmarks.}")
print(r"\label{tab:sota_comparison}")
print(r"\small")
print(r"\begin{tabular}{@{}lcccc@{}}")
print(r"\toprule")
print(r"\textbf{Method} & \textbf{ASV19 LA} & \textbf{ASV21 LA} & \textbf{ASV21 DF} & \textbf{ITW} \\")
print(r"\midrule")
for _, row in display_df.iterrows():
    method = row['Method'].replace('&', r'\&').replace('_', r'\_')
    vals = ' & '.join([row[c] for c in eer_cols])
    if 'Ours' in str(row.get('Source', '')):
        method = r'\textbf{' + method + '}'
    print(f"{method} & {vals} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

## 7. Per-Epoch EER Comparison

Overlay ITW EER curves from multiple training runs to visualize convergence behavior and compare training dynamics across configurations. This helps identify:

- **Convergence speed**: How quickly each configuration reaches its best EER
- **Stability**: Whether EER oscillates or decreases smoothly
- **Overfitting**: Whether EER increases in later epochs (sign of source-domain overfitting)
- **Phase transition effects**: Whether backbone unfreezing causes EER spikes

Each line represents one training run. The legend is placed outside the plot area to avoid occluding data.

In [ ]:
# ============================================================================
# 7.0 Per-Epoch EER Comparison — overlay multiple runs
# ============================================================================

def plot_eer_comparison(runs_dict, metric_key='itw_eer', save_name=None):
    """Overlay EER curves from multiple runs on a single plot.

    Uses flattened history keys (e.g. 'itw_eer', '2019_la_eer').
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    plotted = 0

    for name, run in runs_dict.items():
        history = run.get('history', [])
        if not history:
            continue

        epochs = [h['epoch'] for h in history]
        eers = [h.get(metric_key) for h in history]

        if not any(e is not None for e in eers):
            continue

        eers_clean = [e if e is not None else float('nan') for e in eers]

        # Shorten label
        cfg = run.get('config', {})
        backbone = cfg.get('backbone', '?')
        seed = cfg.get('seed', '?')
        glada = cfg.get('use_glada', '?')
        label = f"{name.split('/')[0]} ({backbone}, s={seed})"

        ax.plot(epochs, eers_clean, label=label, alpha=0.8, linewidth=1.5,
                color=PALETTE[plotted % len(PALETTE)])

        # Mark best epoch
        valid = [(e, ep) for e, ep in zip(eers_clean, epochs) if not np.isnan(e)]
        if valid:
            best_eer, best_ep = min(valid, key=lambda x: x[0])
            ax.scatter([best_ep], [best_eer], marker='*', s=100, zorder=5,
                       color=PALETTE[plotted % len(PALETTE)])
        plotted += 1

    if plotted == 0:
        print(f"No data for metric '{metric_key}' — skipping EER comparison plot.")
        return

    ax.set_xlabel('Epoch')
    ax.set_ylabel('ITW EER (%)')
    ax.set_title('In-the-Wild EER Over Training')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

    if save_name:
        save_path = os.path.join(SAVE_DIR, save_name)
        fig.savefig(save_path)
        print(f"Saved: {save_path}")
    plt.show()


if all_runs:
    plot_eer_comparison(all_runs, save_name='fig_eer_comparison.png')
else:
    print("No runs loaded — EER comparison skipped.")

## 8. Multi-Seed Statistical Analysis

For publication, we report mean EER +/- standard deviation across multiple random seeds per backbone (seeds: 42, 123, 2026). This quantifies variance due to random initialization and data shuffling.

Following best practices in ML evaluation (Bouthillier et al., 2021), we report:

| Statistic | Description | Required Seeds |
|---|---|---|
| Mean | Central estimate of performance | >= 1 |
| Standard deviation | Spread of results across seeds | >= 2 |
| Min / Max | Range of observed performance | >= 2 |
| 95% confidence interval | Interval likely containing the true mean | >= 3 |

**Why multi-seed?** Single-seed results can be misleading -- Dodge et al. (EMNLP 2020) showed that seed variance in fine-tuning can exceed the gap between methods. Reporting mean +/- std is essential for fair comparison.

In [ ]:
# ============================================================================
# 8.0 Multi-Seed Statistical Analysis — per-backbone stats
# ============================================================================

def multi_seed_analysis(runs_dict):
    """Analyze results across multiple seeds for the same backbone.

    Uses flattened history keys ('itw_eer').
    """
    backbone_results = defaultdict(list)

    for name, run in runs_dict.items():
        config = run.get('config', {})
        backbone = config.get('backbone', 'unknown')
        history = run.get('history', [])
        if not history:
            continue

        # Best ITW EER from flattened history
        itw_eers = [h.get('itw_eer') for h in history if h.get('itw_eer') is not None]
        best_eer = min(itw_eers) if itw_eers else None

        if best_eer is not None:
            backbone_results[backbone].append({
                'seed': config.get('seed', 42),
                'best_eer': best_eer,  # Already in %
                'run': name,
                'epochs': len(history),
            })

    if not backbone_results:
        print("No multi-seed data available.")
        return None

    print("Multi-Seed Statistical Analysis")
    print("=" * 70)

    summary_rows = []
    for backbone in sorted(backbone_results.keys()):
        results = backbone_results[backbone]
        eers = [r['best_eer'] for r in results]
        n = len(eers)

        print(f"\n{backbone} ({n} seed{'s' if n > 1 else ''}):")
        for r in sorted(results, key=lambda x: x['seed']):
            print(f"  Seed {r['seed']:>5}: {r['best_eer']:6.2f}% ({r['epochs']} epochs)")

        mean_eer = np.mean(eers)
        row = {'Backbone': backbone, 'N': n, 'Mean EER (%)': mean_eer}

        if n >= 2:
            std_eer = np.std(eers, ddof=1)
            row['Std (%)'] = std_eer
            row['Min (%)'] = min(eers)
            row['Max (%)'] = max(eers)
            print(f"  Mean:  {mean_eer:.2f}% \u00b1 {std_eer:.2f}%")
            print(f"  Range: [{min(eers):.2f}%, {max(eers):.2f}%]")

            if n >= 3:
                se = std_eer / np.sqrt(n)
                t_val = stats.t.ppf(0.975, df=n - 1)
                ci_low = mean_eer - t_val * se
                ci_high = mean_eer + t_val * se
                row['95% CI Low'] = ci_low
                row['95% CI High'] = ci_high
                print(f"  95% CI: [{ci_low:.2f}%, {ci_high:.2f}%]")
        else:
            print(f"  (Single seed — need seeds 42, 123, 2026 for stats)")

        summary_rows.append(row)

    df_summary = pd.DataFrame(summary_rows)
    print("\n" + "=" * 70)
    print("Summary Table:")
    print(df_summary.to_markdown(index=False, floatfmt='.2f'))
    return df_summary


if all_runs:
    stats_df = multi_seed_analysis(all_runs)
else:
    print("No runs loaded — multi-seed analysis skipped.")

## 9. Generate All Thesis Figures

Batch-generate all figures needed for the thesis report and defense presentation. All saved at 300 DPI to the figures directory on Drive.

**Generated files:**

| Filename | Description | Thesis Section |
|---|---|---|
| `fig_training_curves.png` | Training dynamics of best model (2x3 grid) | Results |
| `fig_backbone_comparison.png` | SSL backbone comparison bar chart | Backbone Selection |
| `fig_eer_comparison.png` | Per-epoch EER convergence overlay | Training Analysis |
| `ablation_chart.png` | Ablation study horizontal bar chart | Ablation Study |
| `experiment_summary.json` | Machine-readable summary of all results | (LaTeX automation) |

**Note:** Figures use `savefig.dpi=300` (set in rcParams) for print quality. Captions are added in LaTeX, not in the figure titles -- figure titles shown here are for notebook reference only.

In [ ]:
# ============================================================================
# 9.0 Batch Figure Generation — all thesis figures saved to Drive
# ============================================================================

print("Generating all thesis figures...")
print(f"Save directory: {SAVE_DIR}")
print("=" * 60)

figures_generated = []

# --- 1. Training curves for the best run (lowest ITW EER) ---
if all_runs:
    def get_best_itw(run_data):
        history = run_data.get('history', [])
        itw_eers = [h.get('itw_eer') for h in history if h.get('itw_eer') is not None]
        return min(itw_eers) if itw_eers else 999

    best_run_name = min(all_runs, key=lambda k: get_best_itw(all_runs[k]))
    best_run = all_runs[best_run_name]
    best_cfg = best_run.get('config', {})
    phase_b = best_cfg.get('freeze_epochs', best_cfg.get('phase1_epochs', 5))

    print(f"\n1. Training curves for best run: {best_run_name}")
    plot_training_curves(
        best_run['history'],
        title=f'Training Curves — {best_run_name}',
        save_name='fig_training_curves.png',
        phase_boundary=phase_b,
    )
    figures_generated.append('fig_training_curves.png')

    # --- 2. EER convergence overlay ---
    print("\n2. EER convergence overlay")
    plot_eer_comparison(all_runs, save_name='fig_eer_comparison.png')
    figures_generated.append('fig_eer_comparison.png')
else:
    print("\n1-2. SKIPPED — no training history runs found")

# --- 3. Backbone comparison (from comprehensive eval) ---
if comp_results:
    print("\n3. Backbone comparison")
    plot_backbone_comparison(comp_results, save_name='fig_backbone_comparison.png')
    figures_generated.append('fig_backbone_comparison.png')
else:
    print("\n3. Backbone comparison — SKIPPED (no comprehensive eval)")

# --- 4. Ablation chart + heatmap (generated in Cell 10 above) ---
if comp_results:
    print("\n4. Ablation chart + heatmap (generated in Section 5 above)")
    figures_generated.append('fig_ablation_chart.png')
    figures_generated.append('fig_crossdataset_heatmap.png')
else:
    print("\n4. Ablation chart — SKIPPED (no comprehensive eval)")

# --- Summary ---
print("\n" + "=" * 60)
print(f"{len(figures_generated)} figures saved to {SAVE_DIR}:")
for fname in figures_generated:
    fpath = os.path.join(SAVE_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
    print(f"  {fname} ({size_kb:.0f} KB)")

## 10. Export Summary

Export a machine-readable JSON summary of all experiment results. This can be loaded by the LaTeX build process or used for automated table generation in the thesis.

**Summary structure:**
```json
{
  "generated": "2026-02-16T...",
  "total_runs": N,
  "runs": {
    "run_name": {
      "backbone": "wavlm-large",
      "seed": 42,
      "epochs": 50,
      "best_eer": 0.085,
      "use_glada": true
    }
  }
}
```

In [ ]:
# ============================================================================
# 10.0 Export Summary JSON — machine-readable results for LaTeX integration
# ============================================================================

from datetime import datetime

summary = {
    'generated': datetime.now().isoformat(),
    'total_training_runs': len(all_runs),
    'total_eval_models': len(comp_results),
    'figures_dir': SAVE_DIR,
    'figures_generated': figures_generated if 'figures_generated' in dir() else [],
}

# --- Training run summaries (from history.json) ---
summary['training_runs'] = {}
for name, run in all_runs.items():
    config = run.get('config', {})
    history = run.get('history', [])

    best_eer = None
    best_epoch = None
    if history:
        itw_eers = [(h.get('itw_eer'), h.get('epoch')) for h in history
                     if h.get('itw_eer') is not None]
        if itw_eers:
            best_eer, best_epoch = min(itw_eers, key=lambda x: x[0])

    summary['training_runs'][name] = {
        'backbone': config.get('backbone'),
        'seed': config.get('seed'),
        'epochs': len(history),
        'best_itw_eer': best_eer,
        'best_epoch': best_epoch,
        'use_glada': config.get('use_glada'),
    }

# --- Comprehensive eval summaries ---
summary['comprehensive_eval'] = {}
for name, entry in comp_results.items():
    ds_summary = {}
    for ds_name, ds_data in entry.get('datasets', {}).items():
        ds_summary[ds_name] = {
            'eer': ds_data.get('eer'),
            'ci_low': ds_data.get('ci_low'),
            'ci_high': ds_data.get('ci_high'),
        }
    summary['comprehensive_eval'][name] = {
        'backbone': entry.get('backbone'),
        'datasets': ds_summary,
    }

# Save
summary_path = os.path.join(SAVE_DIR, 'experiment_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to: {summary_path}")
print(f"Training runs: {summary['total_training_runs']}")
print(f"Eval models: {summary['total_eval_models']}")

if summary['training_runs']:
    print("\nTraining runs:")
    for name, info in summary['training_runs'].items():
        eer_str = f"{info['best_itw_eer']:.2f}%" if info['best_itw_eer'] else 'N/A'
        print(f"  {name}: {info['backbone']} | seed={info['seed']} | best_itw={eer_str} @ epoch {info['best_epoch']}")

## Zip up figures and export

In [ ]:
import shutil
import os
from google.colab import files

DRIVE_FIGURES = '/content/drive/MyDrive/Jakob/Experiments/figures'
STAGING = '/content/colab_download/figures'
os.makedirs(STAGING, exist_ok=True)

for f in os.listdir(DRIVE_FIGURES):
    if f.endswith(('.png', '.json')):
        shutil.copy2(os.path.join(DRIVE_FIGURES, f), f'{STAGING}/{f}')
        print(f'Figure: {f}')

shutil.make_archive('/content/colab_export', 'zip', '/content/colab_download')
print(f'\nZip size: {os.path.getsize("/content/colab_export.zip") / 1024:.0f} KB')
files.download('/content/colab_export.zip')